# **VirtualiZarr and grib2io Integration**

This notebook demonstrates how to use **VirtualiZarr** with `grib2io` to create virtual Xarray datasets from GRIB2 files. VirtualiZarr allows you to treat archival files as virtual Zarr stores using familiar Xarray syntax and manipulate them as metadata without loading the data.

In [ ]:
import xarray as xr
import grib2io
from grib2io.kerchunk import ReferenceGenerator
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser
import json
import os

## **1. Generating a Kerchunk Manifest via grib2io**

VirtualiZarr can ingest Kerchunk manifests to build its virtual representation of a dataset. We use `grib2io.kerchunk.ReferenceGenerator` to create this manifest, which contains the necessary byte-range mapping for GRIB2 messages.

In [ ]:
# Path to a sample GRIB2 file
input_data_dir = "../tests/input_data"
grib_file = os.path.join(input_data_dir, "gfs.jpeg.grib2")

# Initialize and generate manifest
gen = ReferenceGenerator(grib_file)
manifest = gen.generate()

# Save to a temporary JSON file for VirtualiZarr to consume
manifest_path = "gfs_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f)

## **2. Ingesting into VirtualiZarr**

Now we use `open_virtual_dataset` with the `KerchunkJSONParser` to load the manifest into VirtualiZarr. This creates an Xarray dataset where data variables are virtual `ManifestArray` objects.

In [ ]:
# Open as a virtual dataset
vds = open_virtual_dataset(
    manifest_path, 
    parser=KerchunkJSONParser(),
    loadable_variables=[]  # Keep everything virtual (don't load coordinates into memory)
)

display(vds)

## **3. Inspecting the Virtual Dataset**

VirtualiZarr datasets look and feel like normal Xarray datasets, but their data variables are backed by `ManifestArray` objects instead of NumPy or Dask arrays. You can inspect the virtual chunks without touching the source GRIB2 file.

In [ ]:
# Check the underlying virtual chunks for a variable
var_name = list(vds.data_vars)[0]
print(f"Variable: {var_name}")
display(vds[var_name].data)  # This is a ManifestArray

## **4. Virtual Concatenation**

One of the most powerful features of VirtualiZarr is the ability to concatenate multiple virtual datasets. This operation only manipulates metadata, making it extremely fast even for massive datasets.

In [ ]:
# Concatenate the dataset with itself as a demonstration
vds_multi = xr.concat([vds, vds], dim="refDate")

print(f"Original refDate size: {vds.refDate.size}")
print(f"Concatenated refDate size: {vds_multi.refDate.size}")
display(vds_multi)

## **5. Exporting to Icechunk via VirtualiZarr**

VirtualiZarr can export its virtual representation to an Icechunk store, allowing you to persist your virtualized datacube.

In [ ]:
import icechunk

icechunk_path = "vds_icechunk_store"
storage = icechunk.local_filesystem_storage(path=icechunk_path)
repo = icechunk.Repository.create(storage)
session = repo.writable_session("main")

# Export the virtual dataset to the Icechunk store using the .vz accessor
vds_multi.vz.to_icechunk(session.store)
session.commit("Exported from VirtualiZarr")

print(f"Virtual dataset exported to {icechunk_path}")